In [0]:


# COMMAND ----------

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, explode, current_timestamp
from delta.tables import DeltaTable
from pyspark.sql.functions import from_json, ArrayType

# COMMAND ----------

# 1. Outer Schema Enforcement
constructors_schema = StructType(fields=[
    StructField("MRData", StructType([
        StructField("ConstructorTable", StructType([
            StructField("Constructors", StringType(), True)
        ]), True)
    ]), True)
])

raw_constructors_df = spark.read \
    .schema(constructors_schema) \
    .json("/mnt/f1-raw/constructors/*")

# COMMAND ----------

# 2. Inner Array Schema Definition & Processing
constructor_element_schema = StructType([
    StructField("constructorId", StringType(), False),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True)
])

parsed_df = raw_constructors_df.withColumn(
    "constructor_array", 
    from_json(col("MRData.ConstructorTable.Constructors"), ArrayType(constructor_element_schema))
)
exploded_df = parsed_df.select(explode(col("constructor_array")).alias("constructor"))

silver_constructors_df = exploded_df.select(
    col("constructor.constructorId").alias("constructor_id"),
    col("constructor.name").alias("team_name"),
    col("constructor.nationality").alias("nationality"),
    current_timestamp().alias("ingestion_date")
).dropDuplicates(["constructor_id"])

display(silver_constructors_df)

# COMMAND ----------

# 3. Idempotent Upsert into Silver Layer
target_path = "dbfs:/mnt/f1-transformed/constructors"

spark.sql("CREATE DATABASE IF NOT EXISTS hive_metastore.f1_transformed")

if not spark.catalog.tableExists("hive_metastore.f1_transformed.constructors"):
    silver_constructors_df.write \
        .format("delta") \
        .option("path", target_path) \
        .mode("overwrite") \
        .saveAsTable("hive_metastore.f1_transformed.constructors")
    print("Constructors table successfully initialized in your Azure container.")
else:
    tgt_table = DeltaTable.forName(spark, target_path)
    tgt_table.alias("tgt") \
        .merge(source = silver_constructors_df.alias("src"), condition = "tgt.constructor_id = src.constructor_id") \
        .whenMatchedUpdate(set = {
            "team_name": "src.team_name", "nationality": "src.nationality", "ingestion_date": "src.ingestion_date"
        }) \
        .whenNotMatchedInsert(values = {
            "constructor_id": "src.constructor_id", "team_name": "src.team_name", 
            "nationality": "src.nationality", "ingestion_date": "src.ingestion_date"
        }) \
        .execute()
    print("Constructors incremental update complete.")